This is a notebook for testing filtering functions.

In [3]:
library(tidyverse)
library(tis)
library(baseballr)
library(hoopR)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘tis’


The following objects are masked from ‘package:lubridate’:

    day, hms, month, period, POSIXct, quarter, today, year, ymd


The following object is masked from ‘package:dplyr’:

    between




In [1]:
setwd('..')
getwd()
# Combine all datasets into one
#source('./code/merge-data.r')

[1] "/accounts/masters/ben_khothsombath/repos/230A-final-project"

In [4]:
od_data <- read_csv("./data/od-data.csv")
head(od_data)

Rows: 67770440 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): origin, destination
dbl  (2): hour, ridership
date (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


date,hour,origin,destination,ridership
<date>,<dbl>,<chr>,<chr>,<dbl>
2018-01-01,0,12TH,12TH,3
2018-01-01,0,12TH,16TH,1
2018-01-01,0,12TH,BAYF,1
2018-01-01,0,12TH,CAST,3
2018-01-01,0,12TH,CIVC,2
2018-01-01,0,12TH,CONC,2


In [6]:
od_complete <- od_data |>
    complete(date, hour, origin, destination, fill = list(ridership = 0))

format(object.size(od_complete), units = "GB")

[1] "6.5 Gb"

In [7]:
head(od_complete)

date,hour,origin,destination,ridership
<date>,<dbl>,<chr>,<chr>,<dbl>
2018-01-01,0,12TH,12TH,3
2018-01-01,0,12TH,16TH,1
2018-01-01,0,12TH,19TH,0
2018-01-01,0,12TH,24TH,0
2018-01-01,0,12TH,ANTC,0
2018-01-01,0,12TH,ASHB,0


Now, let's filter for only operational hours.

In [8]:
dim(od_complete)

[1] 175080000         5

It looks like Drew's original filtering doesn't filter out all hours, so let me adjust the filter to only get operational hours.

In [12]:
od_filtered <- od_complete |>
                    mutate(day = case_when(wday(date) %in% seq(2, 6) ~ "weekday",
                                            wday(date) == 1 ~ "sunday",
                                            wday(date) == 7 ~ "saturday")) |>
                    filter((day == "weekday") & !(hour %in% 0:4) |
                                    (day == "saturday") & !(hour %in% 0:5) | 
                                    (day == "sunday") & !(hour %in% 0:7))

dim(od_filtered)

[1] 134435000         6

In [13]:
format(object.size(od_filtered), units = "GB")

[1] "6 Gb"

Note:

* Hour 0: Midnight-1AM
* Hour 1: 1AM-2AM
* ...
* Hour 23: 11PM-Midnight

In [14]:
head(od_filtered)

date,hour,origin,destination,ridership,day
<date>,<dbl>,<chr>,<chr>,<dbl>,<chr>
2018-01-01,5,12TH,12TH,0,weekday
2018-01-01,5,12TH,16TH,0,weekday
2018-01-01,5,12TH,19TH,0,weekday
2018-01-01,5,12TH,24TH,0,weekday
2018-01-01,5,12TH,ANTC,0,weekday
2018-01-01,5,12TH,ASHB,0,weekday
